# Part 11 — Evaluating RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-11-evaluating-rag/rag_eval.ipynb)

*Replace vibes with numbers: measure a RAG system, and locate which half — retrieval or generation — broke.*

📖 Read the essay: https://www.mefby.com/essays/evaluating-rag

🐍 Companion script: `rag_eval.py`

**What you'll build**

- A tiny store-policy corpus and a transparent **lexical retriever** (your Part 6 search, stripped to its bones).
- **Context recall** — a retrieval metric: did we fetch the chunk that actually holds the answer?
- **Faithfulness** — a generation metric: is every claim in the answer supported by the retrieved context? (LLM-as-a-judge, with a deterministic offline stand-in.)
- A **diagnostic rule** that reads the two metrics in pipeline order and points at the failing stage.
- A small **eval set** with a clean pass, a hallucination, a retrieval miss, and a correct refusal — and the score table that pulls the last two apart.

> **Runs fully offline.** This notebook needs only the Python standard library — no API key, no model download, no network. If an `OPENAI_API_KEY` happens to be set the faithfulness judge shows the real call shape, but it always falls through to a transparent, deterministic stand-in so the output never changes. Real models are used automatically if available; otherwise the fallback keeps every cell runnable.

Previous: **Part 10 — Advanced RAG Architectures**. Next: **Part 12 — RAG in Production**.

## Setup

Everything here is pure standard library: `os` (to peek for an API key without ever calling out) and `re` (for a tiny tokenizer). No numpy required — evaluation is mostly bookkeeping, not linear algebra.

In [ ]:
import os
import re

# Make the offline guarantee explicit: we never depend on a key being present.
print('OPENAI_API_KEY set?', bool(os.getenv('OPENAI_API_KEY')))
print('Setup ready — standard library only.')

## The one idea: a wrong answer has exactly two sources

Hold onto this; it is the spine of the whole part. When a RAG answer is wrong, the fault lives in one of two places:

- **Retrieval failed** — the system never fetched the context that holds the answer.
- **Generation failed** — the right context was sitting in the prompt and the model still answered wrong.

These are different bugs with different fixes, and a single "the answer was bad" verdict cannot tell them apart. So we refuse to score the system with one blended number. We measure the two halves *separately* — retrieval on its own metric, generation on its own metric — and the failing surface reveals itself.

We will measure two of the four core metrics, one per surface:

- **Context recall** (retrieval): did we fetch the chunk that holds the answer?
- **Faithfulness** (generation): is every claim in the answer supported by the retrieved context?

Two metrics are enough to show the shape. A real harness would add context precision, answer relevance, and answer correctness.

## The corpus

We reuse the exact little store-policy corpus the Part 6 app indexed: five short policy chunks. Each gets a stable id, `doc_0` through `doc_4`, because our retrieval metric will work in terms of *which ids came back*.

In [ ]:
CORPUS = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "Shipping fees are non-refundable, and items marked final sale cannot be returned or exchanged.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]
DOCS = [{"id": f"doc_{i}", "text": t} for i, t in enumerate(CORPUS)]

for d in DOCS:
    print(d["id"], '->', d["text"])

## A tiny tokenizer

Both our retriever and our faithfulness stand-in compare *content words*. So we need two helpers: a **tokenizer** that splits text into lowercase word-ish tokens (keeping hyphenated codes like `1-800-returns` intact), and a filter that drops common **stop words** — the `the`, `to`, `of`, `is` that carry no topic and would otherwise inflate every overlap.

In [ ]:
STOP = {
    "a", "an", "the", "to", "of", "for", "and", "or", "is", "are", "was", "be",
    "in", "on", "at", "it", "its", "i", "you", "your", "we", "our", "my", "me",
    "do", "does", "how", "can", "will", "with", "within", "from", "have", "has",
    "if", "that", "this", "they", "them", "their", "us", "what", "about",
}


def tokenize(text):
    # keep hyphenated codes/ids like 'e-4042' or '1-800-returns' intact
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def content_tokens(text):
    return [t for t in tokenize(text) if t not in STOP]


print(tokenize("Call 1-800-RETURNS for a refund."))
print(content_tokens("How many days do I have to return an item?"))

## The retriever

In the real Part 6 app, retrieval is cosine similarity over dense embeddings. Here we use a transparent **lexical (token-overlap) retriever** so the notebook runs standalone with nothing to download: score each chunk by how many content words it shares with the query, return the top-k.

One detail matters for the lesson. A chunk with **zero** overlap is not retrieved at all. That is exactly why a *semantic* query phrased with different words than the document — "repair my earbuds" versus a chunk that says "warranty" and "defects" — comes back empty. That is the dense-versus-sparse gap from Part 7, and we will lean on it to manufacture a retrieval miss.

In [ ]:
def retrieve(query, k=3):
    q = set(content_tokens(query))
    scored = []
    for d in DOCS:
        overlap = len(q & set(content_tokens(d["text"])))
        if overlap > 0:                       # no overlap -> not retrieved at all
            scored.append((overlap, d["id"], d))
    # best overlap first; ties broken by id so the demo is deterministic
    scored.sort(key=lambda s: (-s[0], s[1]))
    return [d for _score, _id, d in scored[:k]]


# A query that shares words with the policy:
hit = retrieve("How many days do I have to return an item for a refund?", k=3)
print('returns refund query ->', [d["id"] for d in hit])

# A semantic query with NO shared words ('repair', 'earbuds' aren't in the warranty chunk):
miss = retrieve("Will you repair my earbuds if they stop working on their own?", k=3)
print('repair earbuds query ->', [d["id"] for d in miss], ' <- empty: the lexical gap')

## Metric 1 (retrieval): context recall

**Context recall** asks: of the information needed to answer, how much did we actually retrieve? It is the metric that catches a retriever coming back empty-handed — the failure no prompt engineering can repair.

Computing recall needs a notion of what the answer *required*. Production tools decompose the **reference answer** into claims and check coverage; our transparent stand-in works at the level of **golden chunks** — the specific chunk id(s) that genuinely answer the question. Recall is then simply: of the golden ids, how many showed up in what we retrieved?

One special case carries a lot of weight. If a question has **no golden chunk** — it is genuinely out of scope — there is nothing to retrieve, so recall is undefined. We return `None` (printed as `n/a`). That `None` is what will later distinguish a *correct refusal* from a *retrieval miss*, even when the two produce the identical "I don't know."

In [ ]:
def context_recall(golden_ids, retrieved):
    if not golden_ids:
        return None                       # out of scope: nothing to retrieve
    got = {d["id"] for d in retrieved}
    hit = sum(1 for g in golden_ids if g in got)
    return hit / len(golden_ids)


got_it = retrieve("How many days do I have to return an item for a refund?", k=3)
print('recall, golden present :', context_recall(["doc_0"], got_it))   # 1.0

missed = retrieve("Will you repair my earbuds if they stop working on their own?", k=3)
print('recall, golden missed  :', context_recall(["doc_4"], missed))   # 0.0

print('recall, out of scope   :', context_recall([], got_it))          # None -> n/a

## Metric 2 (generation): faithfulness, the intended shape

**Faithfulness** (also called **groundedness**) is the anti-hallucination metric. It asks: is every claim in the answer actually supported by the retrieved context? The recipe is to break the answer into **atomic claims** and check each one against the context — a small entailment test, sometimes called **NLI** (natural language inference): does the context entail this claim? The score is the fraction of claims that hold up.

At scale you do this with an **LLM-as-a-judge**: hand a capable model the context and the answer, give it a rubric, and read back a structured verdict. Here is the rubric we would send. (Judges are biased — they favor longer answers, favor whichever candidate they see first — and inconsistent, so in real life you validate the judge against human labels before trusting it.)

In [ ]:
JUDGE_PROMPT = """You are grading a RAG answer for FAITHFULNESS.
Given the CONTEXT and the ANSWER, list each atomic claim in the answer, then for
each claim decide whether the CONTEXT supports it (true) or not (false).
Return faithfulness = (supported claims) / (total claims), a number from 0 to 1.

CONTEXT:
{context}

ANSWER:
{answer}
"""

print(JUDGE_PROMPT.format(context="<retrieved chunks>", answer="<the model's answer>"))

## Breaking an answer into atomic claims

Before we can score support claim-by-claim, we have to *split* the answer into claims. Our stand-in does it the cheap, transparent way: cut on sentence boundaries, then split each sentence on the conjunction `and`. "You have 30 days to return an item, and it must be unused" becomes two claims we can check independently.

In [ ]:
def _split_claims(answer):
    parts = re.split(r"[.!?]\s+", answer)     # sentences
    claims = []
    for p in parts:
        claims.extend(s for s in p.split(" and ") if s.strip())  # then conjuncts
    return claims


demo = "You have 30 days from purchase to return an item and it must be unused."
for c in _split_claims(demo):
    print('claim:', c.strip())

## The deterministic faithfulness stand-in

We cannot call an LLM offline, so we mimic the judge's *interface* with zero dependencies. For each claim, count what fraction of its content words appear in the context; call the claim **supported** when most of them do (threshold 0.6). The score is supported claims over total claims — the same shape a real judge returns.

It is crude — a hash of word overlap, not real entailment — but it is honest about what it is, and it catches the invented "hotline" / "instant refund" style hallucination, where the answer's content words simply are not in the retrieved policy.

Two guards make the metric behave correctly:

- An honest **refusal** ("I don't know") invents nothing, so it scores a perfect 1.0 — there are no claims to verify.
- A claim made of only stop words contributes nothing and is skipped.

In [ ]:
REFUSALS = ("don't know", "do not know", "cannot find", "couldn't find")


def _fallback_faithfulness(answer, context):
    """Deterministic stand-in: a claim is 'supported' when most of its content
    words appear in the context. Returns (score, per-claim verdicts)."""
    low = answer.lower()
    if any(p in low for p in REFUSALS):       # an honest refusal invents nothing
        return 1.0, ["(refusal: no claims to verify)"]
    ctx = set(content_tokens(context))
    claims = _split_claims(answer)
    verdicts, supported = [], 0
    for c in claims:
        toks = content_tokens(c)
        if not toks:
            continue
        frac = sum(1 for t in toks if t in ctx) / len(toks)
        ok = frac >= 0.6                      # most of the claim is grounded
        supported += ok
        verdicts.append(f"[{'ok ' if ok else 'NO '}{frac:.2f}] {c.strip()}")
    score = supported / len(verdicts) if verdicts else 1.0
    return score, verdicts


ctx = CORPUS[1]   # the 'to start a return, email support@example.com' chunk
score, verdicts = _fallback_faithfulness(
    "Call our hotline at 1-800-RETURNS and we will issue an instant refund.", ctx)
print('faithfulness =', score)
for v in verdicts:
    print(' ', v)

## The guarded judge: real LLM if available, stand-in otherwise

Now we wrap the two paths the way every part in this series does it: the **real library call is the headline** (shown so you can read the intended shape), wrapped in `try/except`, and the **transparent stand-in keeps the cell runnable** when there is no key, no package, or no network. We deliberately fall through to the deterministic version even after a successful real call, so this teaching notebook always prints the same numbers.

If you ever see the real branch fail, it prints a clear `[real judge unavailable] -> offline fallback` label rather than crashing — the offline guarantee in action.

In [ ]:
def judge_faithfulness(question, context, answer):
    """Score how grounded `answer` is in `context`. Tries a hosted judge if a key
    is present (shown for shape); otherwise uses the transparent fallback."""
    if os.getenv("OPENAI_API_KEY"):
        try:
            from openai import OpenAI
            client = OpenAI()
            client.chat.completions.create(            # the real call shape
                model="gpt-4o-mini",                   # a cheap judge; check names
                messages=[{"role": "user",
                           "content": JUDGE_PROMPT.format(context=context, answer=answer)}],
                temperature=0,
            )
            # A real harness parses the judge's number here. We fall through to the
            # deterministic version below so this notebook always prints the same demo.
        except Exception as exc:
            print(f"[real judge unavailable: {type(exc).__name__}] -> offline fallback")
    return _fallback_faithfulness(answer, context)


# Faithful answer, grounded in doc_0:
s_ok, _ = judge_faithfulness(
    "How many days do I have to return an item for a refund?",
    CORPUS[0],
    "You have 30 days from purchase to return an item for a refund, "
    "if it is unused and in its original packaging.")
print('faithful answer  ->', round(s_ok, 2))

# Hallucinated answer against the same chunk:
s_bad, _ = judge_faithfulness("How do I start a return?", CORPUS[1],
    "Call our hotline at 1-800-RETURNS and we will issue an instant refund.")
print('invented answer  ->', round(s_bad, 2))

## The evaluation set

Your evaluation is only ever as good as the set you measure against, and a good set **covers the real distribution** — including the case that vibes always forget: out-of-scope questions that *should be refused*.

Our tiny set has four cases, each labelled with its golden chunk id(s) (empty = out of scope) and a reference answer. The `answer` field is a canned stand-in for what the Part 6 `ask(q)` would return, so the demo is deterministic; in a real loop you would call the live app here.

The four cases are chosen to hit each corner once:

1. **Clean pass** — right chunk retrieved, faithful answer.
2. **Hallucination** — right chunk retrieved, but the model invents a hotline and an instant refund.
3. **Retrieval miss** — the warranty chunk exists, but the lexical query shares no words with it, so the app refuses.
4. **Correct refusal** — genuinely out of scope; there is no spec in the corpus to find.

In [ ]:
EVAL_SET = [
    {
        "q": "How many days do I have to return an item for a refund?",
        "golden": ["doc_0"],
        "reference": "30 days from purchase, if the item is unused and in its original packaging.",
        "answer": "You have 30 days from purchase to return an item for a refund, "
                  "if it is unused and in its original packaging.",
    },
    {
        "q": "How do I start a return?",
        "golden": ["doc_1"],
        "reference": "Email support@example.com with your order number.",
        # right chunk retrieved, but the model invents a hotline and an instant refund
        "answer": "Call our hotline at 1-800-RETURNS and we will issue an instant refund.",
    },
    {
        "q": "Will you repair my earbuds if they stop working on their own?",
        "golden": ["doc_4"],   # answerable from the warranty chunk...
        "reference": "Electronics carry a one-year limited warranty for manufacturing defects.",
        # ...but a lexical query with no shared words retrieves nothing, so the app refuses
        "answer": "I don't know based on the provided documents.",
    },
    {
        "q": "What is the battery life of the X1 wireless earbuds?",
        "golden": [],          # genuinely out of scope: no spec in the corpus
        "reference": "Not in the documents; the system should refuse.",
        "answer": "I don't know based on the provided documents.",
    },
]

print(f"{len(EVAL_SET)} cases: pass, hallucination, retrieval miss, correct refusal.")

## From metrics to a diagnosis

Here is where the two metrics stop being a report card and become a *tool*. Read them in pipeline order and they localize the bug:

- **Out of scope?** Then the only right behavior is a refusal. Refused -> correct refusal; answered anyway -> fix generation (it should have refused).
- **Low context recall?** A **retrieval** problem — the answer never reached the model. No prompt change can fix it; go work on the retriever.
- **High recall but low faithfulness?** A **generation** problem — the right context was there and the model still made something up. Leave the retriever alone; fix the prompt, lower the temperature, or use a stronger model.
- **Otherwise** — recall present and faithful — it passes.

Note the order: we check retrieval before generation, because an unfaithful-looking answer built on *missing* context is not the generator's fault.

In [ ]:
def diagnose(recall, faith, refused, out_of_scope):
    if out_of_scope:
        return "correct refusal" if refused else "FIX generation (should refuse)"
    if recall is not None and recall < 0.99:
        return "FIX retrieval"
    if faith < 0.6:
        return "FIX generation (faithfulness)"
    return "pass"


# Sanity-check the rule on the four corners:
print(diagnose(1.0, 1.0, False, False))   # pass
print(diagnose(1.0, 0.0, False, False))   # FIX generation (faithfulness)
print(diagnose(0.0, 1.0, True,  False))   # FIX retrieval
print(diagnose(None, 1.0, True, True))    # correct refusal

## The headline demo: run the loop, read the table

Now we wire the pieces into the offline eval loop and print the score table. The table — not any single number — is the point. For each case: retrieve, join the chunks into one context string, score recall and faithfulness, detect a refusal, flag out-of-scope, and diagnose.

In [ ]:
print("Offline eval over the store-policy app  (k=3 retrieval)\n")
header = f"{'question':46}  {'recall':>6}  {'faith':>6}   verdict"
print(header)
print("-" * len(header))

for case in EVAL_SET:
    retrieved = retrieve(case["q"], k=3)
    context = "\n".join(d["text"] for d in retrieved)

    recall = context_recall(case["golden"], retrieved)
    faith, _verdicts = judge_faithfulness(case["q"], context, case["answer"])
    refused = any(p in case["answer"].lower() for p in REFUSALS)
    out_of_scope = not case["golden"]

    verdict = diagnose(recall, faith, refused, out_of_scope)
    r = " n/a " if recall is None else f"{recall:5.2f}"
    print(f"{case['q'][:46]:46}  {r:>6}  {faith:6.2f}   {verdict}")

## The lesson hiding in the last two rows

Look at rows 3 and 4. They produce the **identical** answer — "I don't know" — and to a vibes check they are indistinguishable. But context recall pulls them apart, and that distinction decides whether you touch the retriever or leave it alone. It is invisible to the eye, and it is the entire reason to measure.

In [ ]:
row3 = EVAL_SET[2]   # earbud repair -> retrieval MISS
row4 = EVAL_SET[3]   # battery life  -> correct REFUSAL

for label, case in (("row 3", row3), ("row 4", row4)):
    retrieved = retrieve(case["q"], k=3)
    recall = context_recall(case["golden"], retrieved)
    print(f"{label}: answer = {case['answer']!r}")
    print(f"        retrieved = {[d['id'] for d in retrieved]}  golden = {case['golden']}  recall = {recall}")

print()
print("Same answer, different diagnosis:")
print("  row 3 had a golden chunk (doc_4) that lexical search never found -> retrieval MISS, fix the retriever.")
print("  row 4 had no golden chunk at all -> correct REFUSAL, leave the retriever alone.")

## The loop is the deliverable

A single score in isolation is meaningless. A faithfulness of 0.82 tells you nothing; a faithfulness that went from 0.82 to 0.88 *on the same set* because you added a reranker is a decision you can defend. So the whole discipline is one loop:

> **Build an eval set, measure a baseline, change one thing, re-measure on the same set, keep it if it improves, revert it if it does not.**

Two phrases do all the work. "Change one thing" is what lets you attribute the result — change three at once and you will never know which helped. And "keep it if it improves" means the only number that matters is the **delta** against your last known-good baseline. To make that concrete, here is the tiny scaffold: collapse a whole run into one summary number you can compare across versions.

In [ ]:
def run_eval(eval_set):
    """Run the loop and return a summary you can compare across versions:
    average faithfulness over in-scope cases, and a verdict tally."""
    faiths, tally = [], {}
    for case in eval_set:
        retrieved = retrieve(case["q"], k=3)
        context = "\n".join(d["text"] for d in retrieved)
        recall = context_recall(case["golden"], retrieved)
        faith, _ = judge_faithfulness(case["q"], context, case["answer"])
        refused = any(p in case["answer"].lower() for p in REFUSALS)
        verdict = diagnose(recall, faith, refused, not case["golden"])
        tally[verdict] = tally.get(verdict, 0) + 1
        if case["golden"]:                 # average faith over in-scope cases only
            faiths.append(faith)
    avg_faith = sum(faiths) / len(faiths) if faiths else 0.0
    return {"avg_faithfulness": round(avg_faith, 3), "verdicts": tally}


baseline = run_eval(EVAL_SET)
print('baseline:', baseline)
print()
print("Next version? Change ONE thing, run_eval again, and compare the delta")
print("against this baseline. A number you can defend, not a vibe.")

## What you learned

- **A wrong answer has exactly two sources: retrieval or generation.** The whole purpose of evaluation is to tell which, so you fix the right thing. Measure the two halves separately, not just end to end.
- **Context recall** judges retrieval (did the golden chunk come back?); **faithfulness** judges generation (is every claim grounded in the context?). Low recall points at the retriever; high recall with low faithfulness points at the prompt or model.
- **LLM-as-a-judge** is how you measure faithfulness at scale — but it is biased and inconsistent, so validate it against human labels before you trust it. Here a transparent, deterministic stand-in kept every cell runnable offline.
- **Your evaluation is only as good as your set.** Cover the real distribution, including out-of-scope questions that should be refused — the case vibes always miss.
- **The loop is the deliverable:** build an eval set, measure a baseline, change one thing, re-measure, keep or revert. The only number that matters is the delta.
- The two-refusals row is the whole lesson in miniature: two identical "I don't know" answers, pulled apart by one metric into *fix the retriever* versus *leave it alone*.

**Next: Part 12 — RAG in Production.** The system is smart (Parts 6–10) and now measurable (this part). The last step is keeping it alive under real load: latency, cost, scaling the index, monitoring quality, and security. We tie the whole series together.